# 1 Data download and text extraction

NeMo Curator provides dedicated functions for Common Crawl, Wikipedia, and arXiv, allowing you to immediately begin downloading and extracting text by providing arguments.

(Optional): You can also download and extract text from your own data sources. This requires defining a download class inheriting from NeMo Curator's DocumentDownloader (the same class used in the dedicated functions for Common Crawl, etc.), an iteration class inheriting from DocumentIterator, and a text extraction class inheriting from DocumentExtractor. Examples of these can be found in "Curating Custom Datasets for LLM Parameter-Efficient Fine-Tuning with NVIDIA NeMo Curator" and "Curating Custom Datasets for LLM Training with NVIDIA NeMo Curator," so please refer to them.

This tutorial will download data from the Japanese Wikipedia and extract text from it.

When targeting the Japanese Wikipedia, use the pre-defined function `download_wikipedia()`, specifying Japanese with `language='ja'` and the date of the snapshot to download with `dump_date`. Here, to save time and resources, we set `url_limit` to limit the number of files downloaded (comment this out if you want to download all files).

Adjust the number of workers and memory limits in the `LocalCluster` arguments to suit your environment. In our test environment and with the following settings, this process took approximately 2 hours (this step is the most time-consuming part of this tutorial).

The following scripts after container startup assume cell execution in Jupyter Notebook. Also, each step can be executed individually by changing the path if the dataset used as input already exists.


# 1 データのダウンロードとテキストの抽出
NeMo Curator では、Common Crawl、Wikipedia、arXiv について、それぞれ専用の関数が用意されており、引数を与えて実行することですぐにダウンロードおよびテキストの抽出を開始できます。

(オプション): 独自のデータ ソースをダウンロード、テキスト抽出することも可能です。この処理には、Common Crawl 用などの専用関数でも使われている NeMo Curator の DocumentDownloader を継承したダウンロード用のクラス、DocumentIterator を継承した反復処理用のクラス、DocumentExtractor を継承したテキスト抽出用のクラスを定義する必要があります。これらの例は Curating Custom Datasets for LLM Parameter-Efficient Fine-Tuning with NVIDIA NeMo Curator や Curating Custom Datasets for LLM Training with NVIDIA NeMo Curator に記載がありますのでぜひ参考にしてください。

このチュートリアルでは、日本語 Wikipedia のデータをダウンロードして、そこからテキストを抽出します。

日本語 Wikipedia を対象にする際は、download_wikipedia() というあらかじめ用意された関数を使用し、language='ja' によって日本語を対象に指定、dump_date にいつの時点のスナップショットをダウンロードするか指定します。ここでは時間およびリソースを節約するため、ダウンロードするファイル数を制限する url_limit を設定します (全てのファイルを対象にしたい場合はコメントアウトしてください)。

LocalCluster の引数にあるワーカー数やメモリ制限は実行する環境に合わせて変更してください。今回の検証環境と以下の設定ではこの処理に 2 時間ほど要しました（このステップは本チュートリアルで最も時間がかかるパートになります）。

以下、コンテナー起動後のスクリプトは Jupyter Notebook 上でのセル実行を想定しています。また、それぞれのステップは入力に使用するデータセットがすでに存在する状況であればパスを変更することで個々に実行することが可能です。

In [2]:
import os
import multiprocessing as mp
mp.set_start_method('fork', force=True)

from nemo_curator.download import download_wikipedia
from dask.distributed import Client, LocalCluster

def safe_dump_date(language):
    import requests
    from bs4 import BeautifulSoup

    url = f"https://dumps.wikimedia.org/{language}wiki/"
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    dates = sorted([
        a.get("href").strip("/")
        for a in soup.find_all("a")
        if a.get("href") and a.get("href").strip("/").isdigit()
    ])

    return dates[-1]

#def main():
cur_dir = os.getcwd()
data_dir = cur_dir

cluster = LocalCluster(
    n_workers=32,
    threads_per_worker=2,
    processes=True,
    memory_limit='16GB',
    nanny=False
)
client = Client(cluster)

download_output_directory = os.path.join(data_dir, "wiki_downloads", "data")

dump_date = safe_dump_date("ja")
print(dump_date)

is_download_really = False # TODO set to True if necessary

# TODO
if is_download_really:
    res = download_wikipedia(
        download_output_directory,
        language='ja',
        dump_date="20251020", #"20240201",   # date fixed 
        url_limit=1
    ).df.compute()
    # https://dumps.wikimedia.org/jawiki/20251020

    print("Download task submitted")
    print(res)

#client.close()
#cluster.close()


#if __name__ == "__main__":
#    main()



/usr/local/lib/python3.10/dist-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45907 instead
  warnings.warn(


20260401


2026-04-14 23:10:29,814 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:29,835 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:29,854 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:29,878 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:29,922 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:29,963 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:29,977 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:30,011 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:30,042 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:30,055 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:30,109 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:30,125 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:30,137 - distributed.nanny - WARNING - Restarting worker
2026-04-14 23:10:30,152 - distributed.

In [4]:
! head -n 1 ./wiki_downloads/data/jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2.10lines.jsonl

{"text":"アンパサンド（&, ）は、並立助詞「…と…」を意味する記号である。ラテン語で「…と…」を表す接続詞 \"et\" の合字を起源とする。現代のフォントでも、Trebuchet MS など一部のフォントでは、\"et\" の合字であることが容易にわかる字形を使用している。\n\n語源 \n\n英語で教育を行う学校でアルファベットを復唱する場合、その文字自体が単語となる文字（\"A\", \"I\", かつては \"O\" も）については、伝統的にラテン語の per se（それ自体）を用いて \"A per se A\" のように唱えられていた。また、アルファベットの最後に、27番目の文字のように \"&\" を加えることも広く行われていた。\"&\" はラテン語で et と読まれていたが、のちに英語で and と読まれるようになった。結果として、アルファベットの復唱の最後は \"X, Y, Z, and per se and\" という形になった。この最後のフレーズが繰り返されるうちに \"ampersand\" となまっていき、この言葉は1837年までには英語の一般的な語法となった。\n\nアンドレ＝マリ・アンペールがこの記号を自身の著作で使い、これが広く読まれたため、この記号が \"Ampère's and\" と呼ばれるようになったという誤った語源俗説がある。\n\n歴史 \n\nアンパサンドの起源は1世紀の古ローマ筆記体にまでさかのぼることができる。古ローマ筆記体では、E と T はしばしば合字として繋げて書かれていた（左図「アンパサンドの変遷」の字形1）。それに続く、流麗さを増した新ローマ筆記体では、さまざまな合字が極めて頻繁に使われるようになった。字形2と3は4世紀中頃における et の合字の例である。その後、9世紀のカロリング小文字体に至るラテン文字の変遷の過程で、合字の使用は一般には廃れていった。しかし、et の合字は使われ続け、次第に元の文字がわかりにくい字形に変化していった（字形4から6）。\n\n現代のイタリック体のアンパサンドは、ルネサンス期に発展した筆記体での et の合字にさかのぼる。1455年のヨーロッパにおける印刷技術の発明以降、印刷業者はイタリック体とローマ筆記体のアンパサンドの両方を多用するようになった。アンパサンド

# 2 Language Detection and Isolation, Text Reformatting

In this section, we will classify the documents extracted earlier by language using fasttext's language identification model. This will sort the documents into subfolders created for each language.

# 2 言語の検出と分離、テキストの再フォーマット化

このセクションでは、先ほど抽出したドキュメントを fasttext の言語識別モデルを使用して、言語ごとに分類します。これによって、ドキュメントが言語ごとに作成されるサブフォルダーへ振り分けられます。

In [3]:
import os
import time
from dask.distributed import Client, LocalCluster

from nemo_curator import ScoreFilter,Modify
from nemo_curator.datasets import DocumentDataset
from nemo_curator.filters import FastTextLangId
from nemo_curator.modifiers import UnicodeReformatter
from nemo_curator.utils.file_utils import separate_by_metadata

#def main():

cur_dir = os.getcwd()
print(cur_dir)
data_dir = f"{cur_dir}/"

# 前の処理でclusterを落としている場合は以下をアンコメントして再度起動してください
cluster = LocalCluster(n_workers=48, processes=True, memory_limit='24GB')
client = Client(cluster)

# Input path
#multilingual_data_path = "./wiki_downloads/data/jawiki-20240801-pages-articles-multistream1.xml-p1p114794.bz2.jsonl"
#multilingual_data_path = "./wiki_downloads/data/jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2.jsonl"
multilingual_data_path = "./wiki_downloads/data/jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2.10lines.jsonl"

# Output path
language_base_output_path = os.path.join(data_dir,"language_sep") # /workspace/asr/brev.nemo.curator.20260324/language_sep
language_data_output_path = os.path.join(language_base_output_path,"data") # /workspace/asr/brev.nemo.curator.20260324/language_sep/data
language_separated_output_path = os.path.join(language_data_output_path,"language") # /workspace/asr/brev.nemo.curator.20260324/language_sep/data/language

# Fasttext model path
model_path = language_base_output_path # '/workspace/asr/brev.nemo.curator.20260324/language_sep'

# Define key in output .jsonl files to store the language information
language_field = "language"

#!wget https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin -P {model_path}
t0 = time.time()

# Load dataset 
multilingual_dataset = DocumentDataset.read_json(multilingual_data_path, add_filename=True) # <nemo_curator.datasets.doc_dataset.DocumentDataset object at 0x7fdbb2ad42e0>

# Define Language separation pipeline
lang_filter = FastTextLangId(os.path.join(model_path,'lid.176.bin')) # <nemo_curator.filters.classifier_filter.FastTextLangId object at 0x7fd7e0d947c0>
language_id_pipeline = ScoreFilter(lang_filter, score_field=language_field, score_type='object')

filtered_dataset = language_id_pipeline(multilingual_dataset)

# The language separation pipeline will produce a result looks like ['JA',0.96873], we only want to keep the 'JA' label and drop the detailed classifier score
filtered_dataset.df[language_field] = filtered_dataset.df[language_field].apply(lambda score: score[1],meta = (language_field, 'object'))

# Split the dataset to corresponding language sub-folders
language_stats = separate_by_metadata(filtered_dataset.df, language_separated_output_path, metadata_field=language_field).compute()

print(f"Time taken for splitting language:{time.time()-t0}")

#client.close()
#cluster.close()

#if __name__ == "__main__":
#    main()

/workspace/asr/brev.nemo.curator.20260324/data_curator


/usr/local/lib/python3.10/dist-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36583 instead
  warnings.warn(


Reading 1 files


/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Time taken for splitting language:22.473998308181763


2026-04-14 23:10:23,205 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:34569'.
2026-04-14 23:10:23,211 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:44891'.
2026-04-14 23:10:23,216 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:34745'.
2026-04-14 23:10:23,218 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:37519'.
2026-04-14 23:10:23,221 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:39625'.
2026-04-14 23:10:23,226 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:39159'.
2026-04-14 23:10:23,230 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:33701'.
2026-04-14 23:10:23,232 - distributed.scheduler - WARNING - Received heartbeat from unregistered 

In [5]:
! head -n 1 ./language_sep/data/language/JA/jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2.10lines.jsonl

{"filename":"jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2.10lines.jsonl","id":"5","language":"JA","source_id":"jawiki-20251020-jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2","text":"アンパサンド（&, ）は、並立助詞「…と…」を意味する記号である。ラテン語で「…と…」を表す接続詞 \"et\" の合字を起源とする。現代のフォントでも、Trebuchet MS など一部のフォントでは、\"et\" の合字であることが容易にわかる字形を使用している。\n\n語源 \n\n英語で教育を行う学校でアルファベットを復唱する場合、その文字自体が単語となる文字（\"A\", \"I\", かつては \"O\" も）については、伝統的にラテン語の per se（それ自体）を用いて \"A per se A\" のように唱えられていた。また、アルファベットの最後に、27番目の文字のように \"&\" を加えることも広く行われていた。\"&\" はラテン語で et と読まれていたが、のちに英語で and と読まれるようになった。結果として、アルファベットの復唱の最後は \"X, Y, Z, and per se and\" という形になった。この最後のフレーズが繰り返されるうちに \"ampersand\" となまっていき、この言葉は1837年までには英語の一般的な語法となった。\n\nアンドレ＝マリ・アンペールがこの記号を自身の著作で使い、これが広く読まれたため、この記号が \"Ampère's and\" と呼ばれるようになったという誤った語源俗説がある。\n\n歴史 \n\nアンパサンドの起源は1世紀の古ローマ筆記体にまでさかのぼることができる。古ローマ筆記体では、E と T はしばしば合字として繋げて書かれていた（左図「アンパサンドの変遷」の字形1）。それに続く、流麗さを増した新ローマ筆記体では、さまざまな合字が極めて頻繁に使われるようになった。字形2と3は4世紀中頃における et の合字の例である。その後、9世紀のカロリング小文字体

## Note that "language":"JA" is added to each line of data in the file. That is, the text is of Japanese language.


# 3 Unicode Processing

Once this process is complete (1-2 minutes in the testing environment), the documents will be saved under language_sep/data/language/, categorized by language.

Next, the Unicode in the Japanese documents, which were previously sorted by language, will be reformatted using UnicodeReformatter (which runs ftfy internally).

# 3 Unicode処理

この処理が完了 (検証環境では 1-2 分) すると language_sep/data/language/ の下に分類された言語ごとにドキュメントが保存されています。

次に、先ほど言語ごとに振り分けられたドキュメントの日本語を対象に、UnicodeReformatter (内部で ftfy を実行) を使用して、ドキュメント内のユニコードを再フォーマットします。

In [6]:
from nemo_curator import ScoreFilter,Modify
import time
import os
from nemo_curator.datasets import DocumentDataset
from nemo_curator.modifiers import UnicodeReformatter

t0 = time.time()

# Define desired language
target_language = "JA"

# Output path
cur_dir = os.getcwd()

data_dir = f"{cur_dir}/"
language_base_output_path = os.path.join(data_dir,"language_sep") # /workspace/asr/brev.nemo.curator.20260324/language_sep
language_data_output_path = os.path.join(language_base_output_path,"data") # /workspace/asr/brev.nemo.curator.20260324/language_sep/data
language_separated_output_path = os.path.join(language_data_output_path,"language") # /workspace/asr/brev.nemo.curator.20260324/language_sep/data/language

lang_sep_cleaned_data_output_path = os.path.join(language_data_output_path, "cleaned")

# Read the language specific data and fix the unicode in it
lang_data_path = os.path.join(language_separated_output_path, target_language)
lang_data = DocumentDataset.read_json(lang_data_path,add_filename=True)

cleaner = Modify(UnicodeReformatter())
cleaned_data = cleaner(lang_data)

# Write the cleaned_data
cleaned_data.to_json(lang_sep_cleaned_data_output_path, write_to_filename=True)

print(f"Time taken for fixing unicode:{time.time()-t0}")

'''
root@933d37897f65:/workspace/asr/brev.nemo.curator.20260324# python 3.ftfy.py
/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Reading 1 files
Writing to disk complete for 1 partitions
Time taken for fixing unicode:461.67655992507935
'''


Reading 1 files


/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Writing to disk complete for 1 partitions
Time taken for fixing unicode:17.454552173614502


'\nroot@933d37897f65:/workspace/asr/brev.nemo.curator.20260324# python 3.ftfy.py\n/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.\n  warnings.warn(\nReading 1 files\nWriting to disk complete for 1 partitions\nTime taken for fixing unicode:461.67655992507935\n'

In [7]:
! head -n 1 ./language_sep/data/cleaned/jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2.10lines.jsonl

{"filename":"jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2.10lines.jsonl","id":"5","language":"JA","source_id":"jawiki-20251020-jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2","text":"アンパサンド(&, )は、並立助詞「…と…」を意味する記号である。ラテン語で「…と…」を表す接続詞 \"et\" の合字を起源とする。現代のフォントでも、Trebuchet MS など一部のフォントでは、\"et\" の合字であることが容易にわかる字形を使用している。\n\n語源 \n\n英語で教育を行う学校でアルファベットを復唱する場合、その文字自体が単語となる文字(\"A\", \"I\", かつては \"O\" も)については、伝統的にラテン語の per se(それ自体)を用いて \"A per se A\" のように唱えられていた。また、アルファベットの最後に、27番目の文字のように \"&\" を加えることも広く行われていた。\"&\" はラテン語で et と読まれていたが、のちに英語で and と読まれるようになった。結果として、アルファベットの復唱の最後は \"X, Y, Z, and per se and\" という形になった。この最後のフレーズが繰り返されるうちに \"ampersand\" となまっていき、この言葉は1837年までには英語の一般的な語法となった。\n\nアンドレ=マリ・アンペールがこの記号を自身の著作で使い、これが広く読まれたため、この記号が \"Ampère's and\" と呼ばれるようになったという誤った語源俗説がある。\n\n歴史 \n\nアンパサンドの起源は1世紀の古ローマ筆記体にまでさかのぼることができる。古ローマ筆記体では、E と T はしばしば合字として繋げて書かれていた(左図「アンパサンドの変遷」の字形1)。それに続く、流麗さを増した新ローマ筆記体では、さまざまな合字が極めて頻繁に使われるようになった。字形2と3は4世紀中頃における et の合字の例である。その後、9世紀のカロリング小文字体

# 4 Assigning IDs
Japanese Wikipedia data already has IDs, but assigning a unified ID format like <prefix>_<id> makes it easier to identify which document in which dataset has been deleted when working with multiple datasets. This is useful when performing tasks such as deduplication and filtering.

The function that performs this process is AddID(). The arguments for this function are as follows:

id_field: The field is added to the input JSON file. If the key already exists in the JSON file, its value is replaced.

id_prefix: The prefix used for the ID. The default is "doc_id".

start_index: The starting index for the ID. The default is "None". Setting it to "None" uses an unordered ID scheme for faster calculation. Here, it is set to "0" for easier referencing.

# 4 ID の付与
日本語 Wikipedia データにはすでに id がありますが、<prefix>_<id> フォーマットのように統一した id を付与すると複数のデータセットを扱う際に、どのデータセットのどのドキュメントが削除されたのかがわかりやすくなります。これは重複排除やフィルタリングなどを行う際に役に立ちます。

この処理を実行する関数が AddID() です。この関数の引数は以下の通りです。

id_field: フィールドが入力 json ファイルに追加されます。キーが jsonl にすでに存在する場合は、その値が置き換えられます。
id_prefix: ID で使用される接頭辞。デフォルトは「doc_id」です。
start_index: ID の開始インデックス。デフォルトは「None」です。「None」に設定すると、高速な計算のために順序のない ID スキームが使用されます。ここでは、参照を容易にするために「0」に設定しています。

In [8]:
import os
import time

from nemo_curator import AddId
from nemo_curator.datasets import DocumentDataset

from dask.distributed import Client, LocalCluster

#def main():
# 前の処理でclusterを落としている場合は以下をアンコメントして再度起動してください
cluster = LocalCluster(n_workers=24, processes=True, memory_limit='24GB')
client = Client(cluster)

cur_dir = os.getcwd()
data_dir = f"{cur_dir}/"

# Input
add_id_input_data_dir = "./language_sep/data/cleaned"

# Output
added_id_output_path = os.path.join(data_dir, "add_id/cleaned")

# Format of output ID will be <prefix>_<id>, Define prefix here
add_ID_id_prefix="JA_wiki"

t0 = time.time()
# Read input files
dataset = DocumentDataset.read_json(add_id_input_data_dir,add_filename=True)

# Run AddID() on the input dataset
add_id = AddId(id_field='id',id_prefix=add_ID_id_prefix,start_index=0)
id_dataset = add_id(dataset)

# Output files
id_dataset.to_json(added_id_output_path, write_to_filename=True)

print(f"Time taken for add ID:{time.time()-t0}")

#client.cluster.close()
#client.shutdown()
#cluster.close()

#if __name__ == "__main__":
#    main()

/usr/local/lib/python3.10/dist-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34463 instead
  warnings.warn(


Reading 1 files


/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Writing to disk complete for 1 partitions
Time taken for add ID:17.8375563621521


In [11]:
print(os.getcwd())

/workspace/asr/brev.nemo.curator.20260324/data_curator


In [15]:
with open("add_id/cleaned/jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2.10lines.jsonl", "r") as f:
    for _ in range(1):
        print(f.readline())

{"filename":"jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2.10lines.jsonl","id":"JA_wiki-0000000000","language":"JA","source_id":"jawiki-20251020-jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2","text":"アンパサンド(&, )は、並立助詞「…と…」を意味する記号である。ラテン語で「…と…」を表す接続詞 \"et\" の合字を起源とする。現代のフォントでも、Trebuchet MS など一部のフォントでは、\"et\" の合字であることが容易にわかる字形を使用している。\n\n語源 \n\n英語で教育を行う学校でアルファベットを復唱する場合、その文字自体が単語となる文字(\"A\", \"I\", かつては \"O\" も)については、伝統的にラテン語の per se(それ自体)を用いて \"A per se A\" のように唱えられていた。また、アルファベットの最後に、27番目の文字のように \"&\" を加えることも広く行われていた。\"&\" はラテン語で et と読まれていたが、のちに英語で and と読まれるようになった。結果として、アルファベットの復唱の最後は \"X, Y, Z, and per se and\" という形になった。この最後のフレーズが繰り返されるうちに \"ampersand\" となまっていき、この言葉は1837年までには英語の一般的な語法となった。\n\nアンドレ=マリ・アンペールがこの記号を自身の著作で使い、これが広く読まれたため、この記号が \"Ampère's and\" と呼ばれるようになったという誤った語源俗説がある。\n\n歴史 \n\nアンパサンドの起源は1世紀の古ローマ筆記体にまでさかのぼることができる。古ローマ筆記体では、E と T はしばしば合字として繋げて書かれていた(左図「アンパサンドの変遷」の字形1)。それに続く、流麗さを増した新ローマ筆記体では、さまざまな合字が極めて頻繁に使われるようになった。字形2と3は4世紀中頃における et の合字の例である。

In [19]:
# client.cluster.close()
# client.shutdown()

## Note that "id":"JA_wiki-0000000000" is added to each text in the file.

# 5 Duplicate Removal

Duplicate removal supports Exact Deduplication, Fuzzy Deduplication, and Semantic Deduplication using embeddings. This section covers Exact Deduplication and Fuzzy Deduplication. For information on Semantic Deduplication, please refer to this section.

Note: To perform these operations, the IDs in the corpus must be unique, for example, by using AddID() in the previous section.

Exact Deduplication
Exact Deduplication hashes the document text into a unique string using a specific hash algorithm, such as "md5". Documents with strictly identical hash values ​​have identical text.

The function used here is ExactDuplicates(). This function has the following arguments:

id_field: The key in the input file to identify the document ID
text_field: The key in the input file containing the document text
hash_method: The hash algorithm to use. The default is md5.
cache_dir: If specified, duplicate document IDs are output to cache_dir. If not specified, IDs are not saved.
Also, use a GPU Dask cluster to speed up deduplication calculations.

# 5 重複排除

重複排除では、Exact Deduplication, Fuzzy Deduplication, 埋め込みを使用した Semantic Deduplication がサポートされています。ここでは、Exact Deduplication と Fuzzy Deduplication を扱います。Semantic Deduplication については、こちらを参照してください。

Note: これらの処理を実行するには、前セクションの AddID() などを使用して、コーパス内の id が一意の状態になっている必要があります。

Exact Deduplication
Exact Deduplication では、ドキュメントのテキストは「md5」などの特定のハッシュ アルゴリズムを使用して一意の文字列にハッシュ化されます。厳密に同一ハッシュ値を持つドキュメントは、同一のテキストを持っています。

ここで使用する関数は ExactDuplicates() です。この関数の引数には、以下のものがあります。

id_field: ドキュメント ID を識別するための入力ファイル内のキー
text_field: ドキュメントのテキストを含む入力ファイル内のキー
hash_method: 使用されるハッシュ アルゴリズム。デフォルトは md5
cache_dir: 指定された場合、重複したドキュメント ID は cache_dir に出力されます。指定されていない場合は、ID は保存されません
また、GPU の dask クラスターを使用して、重複排除の計算を高速化します。

In [20]:
#%env DASK_DATAFRAME__QUERY_PLANNING=False

import os
import time
import pandas as pd

import os
os.environ["DASK_DATAFRAME__QUERY_PLANNING"] = "False"

from nemo_curator.datasets import DocumentDataset
from nemo_curator.modules import ExactDuplicates
from nemo_curator.utils.distributed_utils import get_client, get_num_workers

#def main():

def pre_imports():
    import cudf 


cur_dir = os.getcwd()
data_dir = f"{cur_dir}/"


client = get_client(cluster_type='gpu', set_torch_to_use_rmm=False)
print(f"Number of dask worker:{get_num_workers(client)}")
client.run(pre_imports)

# Input
exact_dedup_input_dataset_dir = "./add_id/cleaned"

# Output
exact_dedup_base_output_path = os.path.join(data_dir, "exact_dedup")
exact_dedup_log_dir = os.path.join(exact_dedup_base_output_path, 'log')
exact_dedup_output_dir = os.path.join(exact_dedup_base_output_path, 'data')

# Parameters for ExactDuplicates()
exact_dedup_dataset_id_field = "id"
exact_dedup_dataset_text_field = "text"

#!mkdir -p {exact_dedup_log_dir}
#!mkdir -p {exact_dedup_output_dir}

t0 = time.time()
# Read input dataset
input_dataset = DocumentDataset.read_json(exact_dedup_input_dataset_dir, backend='cudf')

# Run exact deduplication to the input
exact_dup = ExactDuplicates(
    logger=exact_dedup_log_dir,
    id_field=exact_dedup_dataset_id_field,
    text_field=exact_dedup_dataset_text_field,
    hash_method="md5",
    cache_dir=exact_dedup_output_dir  # Duplicated document ID list is output to the cache_dir
)
duplicates = exact_dup(dataset=input_dataset)

print(f"Number of exact duplicated file:{len(duplicates)}")
print(f"Time taken for exact duplicate:{time.time()-t0}")

exact_dedup_res = pd.read_parquet(os.path.join(exact_dedup_output_dir, "_exact_duplicates.parquet"))
print(f"Number of exact duplicated document:{len(exact_dedup_res)}")
exact_dedup_res.head()

exact_dedup_res.groupby('_hashes')['id'].agg(lambda x: ' '.join(x)).reset_index().head()

#if __name__ == "__main__":
#    main()


/usr/local/lib/python3.10/dist-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35619 instead
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of 

Number of dask worker:8
Reading 1 files


/opt/NeMo-Curator/nemo_curator/modules/exact_dedup.py:158: UserWarning: Output path f/workspace/asr/brev.nemo.curator.20260324/data_curator/exact_dedup/data/_exact_duplicates.parquet already exists and will be overwritten
  warnings.warn(


Number of exact duplicated file:0
Time taken for exact duplicate:67.95691847801208
Number of exact duplicated document:0


,_hashes,id


2026-04-14 23:10:01,562 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:41897'.
2026-04-14 23:10:01,566 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:42921'.
2026-04-14 23:10:03,652 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:33527'.
2026-04-14 23:10:03,659 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:46245'.
2026-04-14 23:10:03,669 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:38169'.
2026-04-14 23:10:03,676 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:38457'.
2026-04-14 23:10:03,679 - distributed.scheduler - WARNING - Received heartbeat from unregistered worker 'tcp://127.0.0.1:35567'.
2026-04-14 23:10:03,684 - distributed.scheduler - WARNING - Received heartbeat from unregistered 

In [21]:
import pandas as pd

df = pd.read_parquet("exact_dedup/data/_exact_duplicates.parquet/part.0.parquet")
print(df.head())   # 前5行

Empty DataFrame
Columns: [id, _hashes]
Index: []


# 6 Fuzzy Deduplication
Unlike Exact Deduplication, Fuzzy Deduplication does not find exact duplicates, but rather extracts similar text based on text statistics using a GPU-implemented MinhashLSH algorithm (this is different from semantic similarity). There are several intermediate steps to extract these duplicates; see here for details.

While Fuzzy Deduplication can be performed step-by-step individually, it can be easily done using FuzzyDuplicates(). You set the n-gram length, the number of buckets and hashes within those buckets, and the Jaccard similarity threshold for determining duplicates as arguments within FuzzyDuplicatesConfig(), and pass them to FuzzyDuplicates().

Note: Fuzzy Deduplication only works with ID format using AddID() or integer ID format.

# 6 Fuzzy Deduplication
Fuzzy Deduplication は、Exact Deduplication とは異なり、厳密な重複を見つけ出すのではなく、GPU 実装された MinhashLSH アルゴリズムによって、テキストの統計に基づき類似テキストを抽出します (意味的な類似性とは異なります)。この重複を抽出するには、複数の中間ステップがありますが、詳細はこちらを参照してください。

Fuzzy Deduplication は複数の手順を個々にステップバイステップで実行することも可能ですが、FuzzyDuplicates() を使用すると簡単に実行できます。FuzzyDuplicatesConfig() 内の引数として、n-gram の長さ、バケットやバケット内の hash の数、重複と判断する Jaccard 類似度の閾値などを設定し、FuzzyDuplicates() に渡します。

Note: Fuzzy Deduplication は、AddID() を使用した id 形式、もしくは整数 id の形式でのみ機能します。



In [29]:
client.close()


## example

1: "hello world"

2: "hello world!"

3: "hi world"

4: "good morning"

1 and 2 and 3 are similar
4 is different with 1/2/3.

So, we may only want to keep 1/4 and remove 2/3.


bucket A:
  - hello world
  - hello world!

bucket B:
  - good morning
  

Outputs are alike:

id1      id2      score

1        2        0.95
  
  

In [36]:
import os
import pandas as pd
from difflib import SequenceMatcher

# ===== read data =====
df = pd.read_json("./add_id/cleaned/jawiki-20251020-pages-articles-multistream1.xml-p1p114794.bz2.10lines.jsonl", lines=True)

# ===== similarity function =====
def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# ===== pair wise compare=====
results = []

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        sim = similarity(df.loc[i, "text"], df.loc[j, "text"])
        if sim > 0.01:   # TODO fine-tune the threshold value here
            results.append((df.loc[i, "id"], df.loc[j, "id"], sim))

result_df = pd.DataFrame(results, columns=["id1", "id2", "score"])

print(result_df)

                   id1                 id2     score
0   JA_wiki-0000000000  JA_wiki-0000000001  0.027634
1   JA_wiki-0000000000  JA_wiki-0000000003  0.046782
2   JA_wiki-0000000000  JA_wiki-0000000004  0.019042
3   JA_wiki-0000000000  JA_wiki-0000000005  0.043880
4   JA_wiki-0000000000  JA_wiki-0000000007  0.020230
5   JA_wiki-0000000000  JA_wiki-0000000008  0.034446
6   JA_wiki-0000000000  JA_wiki-0000000009  0.025221
7   JA_wiki-0000000001  JA_wiki-0000000003  0.032223
8   JA_wiki-0000000001  JA_wiki-0000000004  0.010936
9   JA_wiki-0000000001  JA_wiki-0000000005  0.022996
10  JA_wiki-0000000001  JA_wiki-0000000006  0.017346
11  JA_wiki-0000000001  JA_wiki-0000000007  0.025790
12  JA_wiki-0000000001  JA_wiki-0000000008  0.034426
13  JA_wiki-0000000001  JA_wiki-0000000009  0.029665
14  JA_wiki-0000000002  JA_wiki-0000000003  0.012488
15  JA_wiki-0000000002  JA_wiki-0000000006  0.016954
16  JA_wiki-0000000002  JA_wiki-0000000007  0.011870
17  JA_wiki-0000000003  JA_wiki-0000000004  0.

In [33]:
# ===== must starts from =====

is_fuzzy = False

if is_fuzzy:
    import os
    os.environ["DASK_DATAFRAME__QUERY_PLANNING"] = "False"

    import time
    import pandas as pd
    import dask
    import nemo_curator

    from nemo_curator import FuzzyDuplicates, FuzzyDuplicatesConfig
    from nemo_curator.datasets import DocumentDataset
    from nemo_curator.utils.distributed_utils import get_client, get_num_workers

    # ===== start Dask（CPU！）=====
    client = get_client(cluster_type='cpu')
    print(f"Number of dask worker: {get_num_workers(client)}")

    # ===== path =====
    cur_dir = os.getcwd()
    data_dir = cur_dir

    fuzzy_dedup_data_path = "./add_id/cleaned"



    fuzzy_dedup_base_output_path = os.path.join(data_dir, "fuzzy_wrapper")
    fuzzy_dedup_log_dir = os.path.join(fuzzy_dedup_base_output_path, 'log')
    fuzzy_dedup_cache_dir = os.path.join(fuzzy_dedup_base_output_path, 'cache')
    fuzzy_dedup_output_dir = os.path.join(fuzzy_dedup_base_output_path, 'data')

    # ===== clean and recreate dir =====
    os.system(f"rm -rf {fuzzy_dedup_cache_dir}")

    for p in [
        fuzzy_dedup_base_output_path,
        fuzzy_dedup_log_dir,
        fuzzy_dedup_cache_dir,
        fuzzy_dedup_output_dir,
    ]:
        os.makedirs(p, exist_ok=True)

    # ===== parameters =====
    id_field = 'id'
    text_field = 'text'

    # ===== main logic =====
    with dask.config.set({"dataframe.backend": "pandas"}):   # ✅ modify here 

        t0 = time.time()

        # 👉 use pandas backend
        input_dataset = DocumentDataset.read_json(
            fuzzy_dedup_data_path,
            backend='pandas'
        )

        # 👉 for small datasets
        config = FuzzyDuplicatesConfig(
            cache_dir=fuzzy_dedup_cache_dir,
            id_field=id_field,
            text_field=text_field,
            seed=42,
            char_ngrams=3,          # loose
            num_buckets=1,
            hashes_per_bucket=2,
            num_anchors=1,
            jaccard_threshold=0.1,  # low
        )

        fuzzy = FuzzyDuplicates(
            logger=fuzzy_dedup_log_dir,
            config=config
        )

        duplicates = fuzzy(dataset=input_dataset)

        # ===== 防止空结果 crash =====
        try:
            if duplicates.df.shape[0].compute() > 0:
                duplicates.to_parquet(
                    fuzzy_dedup_output_dir,
                    write_to_filename=False
                )

                result = pd.read_parquet(fuzzy_dedup_output_dir)
                print(result.head())
            else:
                print("⚠️ No duplicates found (ok for small scale dataset)")

        except Exception as e:
            print("⚠️ Skip writing parquet due to:", e)

        print(f"Time taken: {time.time() - t0:.2f}s")

# Generating Synthetic Data

You can also leverage LLM to generate datasets for pre-training and customization. Synthetic data is useful when adapting LLM to low-resource languages/domains or when extracting knowledge from other models. Here, let's run a pipeline to generate synthetic data by accessing the build.nvidia.com API endpoint (you will need to create an account on build.nvidia.com and obtain an API key to begin this section).

This section can be run independently of previous tutorials.

First, prepare the necessary parameters, topics, and templates for accessing the API endpoint and for the interaction between them by doing the following:

# 合成データの生成

LLM を活用して、事前学習やカスタマイズに使用するデータセットを生成することもできます。合成データは、LLM を低リソース言語/ドメインに適応させたり、他のモデルから知識を抽出したりする際などに有用です。ここでは、build.nvidia.com の API エンドポイント にアクセスして合成データを生成するパイプラインを実行してみましょう (このセクションを始めるには build.nvidia.com のアカウントを作成し、APIキーを取得する必要があります)。

このセクションはこれまでのチュートリアルとは依存関係なく実行できます。

まず、以下を実行してAPI エンドポイント へアクセスするための準備と相互の処理に必要なパラメーター、トピック、テンプレートを用意します。

In [38]:
import os
import pandas as pd

from nemo_curator.datasets import DocumentDataset
from dask.distributed import Client, LocalCluster

is_combine = False

if is_combine:
    #def main():

    cluster = LocalCluster(n_workers=8, processes=True, memory_limit='24GB')
    client = Client(cluster)

    cur_dir = os.getcwd()
    data_dir = f"{cur_dir}/"

    # Input
    dataset_dir = "./add_id/cleaned"
    exact_dedup_output_dir="./exact_dedup/data"
    ### fuzzy_dedup_output_dir="./fuzzy_wrapper/data" # TODO

    # Output
    dudped_output_dir = os.path.join(data_dir, "remove_duplicate/result.parquet")

    # Relevant parameters
    input_id_field = 'id'
    id_prefix = "JA_wiki"

    #!mkdir -p {dudped_output_dir}
    acmd = f"mkdir -p {dudped_output_dir}"
    print(acmd)
    os.system(acmd)

    #Load .jsonl dataset (GPUメモリが足りない場合はbackend='pandas'へ変更してください)
    input_dataset = DocumentDataset.read_json(dataset_dir, backend='cudf')

    # Load exact deduplicate result and extract list of duplicated document ID　(GPUメモリが足りない場合はbackend='pandas'へ変更してください)
    exact_duplicates = DocumentDataset.read_parquet(os.path.join(exact_dedup_output_dir, "_exact_duplicates.parquet"), backend='cudf')
    exact_docs_to_remove = exact_duplicates.df.map_partitions(
        lambda x: x[x._hashes.duplicated(keep="first")]
    )

    # Remove the duplicated document from input dataset
    result = input_dataset.df[
        ~input_dataset.df[input_id_field].isin(exact_docs_to_remove[input_id_field].compute())
    ]

    ### fuzzy_dedup_output_dir="./fuzzy_wrapper/data" # TODO
    # Loads result from fuzzy dedup wrapper
    ### fuzzy_duplicates = pd.read_parquet(fuzzy_dedup_output_dir) # TODO

    # Generate list of near duplicate document ID
    ### fuzzy_docs_to_remove = fuzzy_duplicates.drop_duplicates(subset=['group'], keep='first') # TODO

    # Remove near duplicates
    ### result = result[~result[input_id_field].isin(fuzzy_docs_to_remove[input_id_field])] # TODO

    # Save final result to local (backend='pandas'の場合は、write_to_filename=Trueをコメントアウトしてください)
    result.to_parquet(dudped_output_dir, write_to_filename=True)

    res = pd.read_parquet(dudped_output_dir)
    print(f"Length of duplicate removed dataset:{len(res)}")

    #if __name__ == "__main__":
    #    main()


In [39]:
# part.0.orig.parquet
import pandas as pd

df = pd.read_parquet("./remove_duplicate/result.parquet/part.0.orig.parquet")
print(df.head())   # 前5行

                                                              filename  \
__null_dask_index__                                                      
0                    jawiki-20251020-pages-articles-multistream1.xm...   
1                    jawiki-20251020-pages-articles-multistream1.xm...   
2                    jawiki-20251020-pages-articles-multistream1.xm...   
3                    jawiki-20251020-pages-articles-multistream1.xm...   
4                    jawiki-20251020-pages-articles-multistream1.xm...   

                                     id language  \
__null_dask_index__                                
0                    JA_wiki-0000000000       JA   
1                    JA_wiki-0000000001       JA   
2                    JA_wiki-0000000002       JA   
3                    JA_wiki-0000000003       JA   
4                    JA_wiki-0000000004       JA   

                                                             source_id  \
__null_dask_index__                       

In [41]:
import json

with open("./remove_duplicate/result.parquet/train.jsonl", "w") as f:
    for _, row in df.iterrows():
        item = {
            "input": row["text"],
            #"output": row["label"]   # 或你自己的字段
        }
        f.write(json.dumps(item) + "\n")

In [42]:
# https://developer.nvidia.com/ja-jp/blog/how-to-use-continual-pre-training-with-japanese-language-on-nemo-framework/ for continue training

2026-04-14 23:53:58,794 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/distributed/core.py", line 1564, in _connect
    comm = await connect(
  File "/usr/local/lib/python3.10/dist-packages/distributed/comm/core.py", line 377, in connect
    handshake = await comm.read()
  File "/usr/local/lib/python3.10/dist-packages/distributed/comm/tcp.py", line 225, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
asyncio.exceptions.CancelledError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/distributed/core.py", line 1674, in connect
    return connect_attempt.result()
asyncio.exceptions.CancelledError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-pac